# 🧬 Cellpose 3D Pipeline — Complete Cell Tracking Solution

**Competition:** [Biohub — Cell Tracking During Development](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development) ($60K Prize)

**Score:** `edge_jaccard + 0.1 × division_jaccard`

---

### 🏗️ Architecture

```
Zarr Volume (T, Z, Y, X)
    │
    ├── 1. Cellpose 3D Segmentation (cyto3 model, GPU)
    │       └── Fallback: Gaussian blob detector
    │
    ├── 2. Post-processing
    │       ├── Size filtering (50–50K voxels)
    │       └── Sequential relabelling
    │
    ├── 3. Hungarian Bipartite Matching
    │       ├── Distance cost (10 µm cutoff)
    │       ├── Volume similarity cost (0.3 weight)
    │       └── Gap-2 bridging (0.9× penalty)
    │
    ├── 4. Division Detection
    │       ├── Orphan-pair identification
    │       └── Volume conservation scoring
    │
    └── 5. Submission CSV (10 columns)
```

> 💡 **100% self-contained** — No external datasets or packages beyond Kaggle defaults + Cellpose.

---

In [ ]:
# ── Cell 1: Dependencies ─────────────────────────────────────────────────────────
import subprocess, sys

def install_if_missing(package, pip_name=None):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or package])

install_if_missing('cellpose')
install_if_missing('zarr')

print('✅ Dependencies ready')

In [ ]:
# ── Cell 2: Imports & Core Data Structures ───────────────────────────────────────
import csv
import logging
import pathlib
import time
from dataclasses import dataclass, field
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Set, Tuple

import numpy as np
import zarr
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Configuration ─────────────────────────────────────────────────────────────
VOXEL_SIZE_UM = (1.625, 0.40625, 0.40625)  # Z, Y, X
ANISOTROPY = VOXEL_SIZE_UM[0] / VOXEL_SIZE_UM[1]  # ~4.0


@dataclass
class Cell:
    """Detected cell instance at a single frame."""
    id: int
    frame: int
    centroid: np.ndarray
    centroid_um: Optional[np.ndarray]
    volume: float
    features: Dict[str, float] = field(default_factory=dict)

    def distance_to(self, other: 'Cell', use_um: bool = True) -> float:
        a = self.centroid_um if use_um and self.centroid_um is not None else self.centroid
        b = other.centroid_um if use_um and other.centroid_um is not None else other.centroid
        return float(np.linalg.norm(a - b))


@dataclass
class DivisionEvent:
    """Detected cell division."""
    parent_id: int
    parent_frame: int
    child1_id: int
    child2_id: int
    division_frame: int
    confidence: float


def open_volume(path):
    root = zarr.open(str(path), mode='r')
    arr = root['0'] if '0' in root else root
    if len(arr.shape) != 4:
        raise ValueError(f'Expected 4D volume, got shape {arr.shape}')
    return arr


def iter_frames(path) -> Iterator[Tuple[int, np.ndarray]]:
    vol = open_volume(path)
    for t in range(vol.shape[0]):
        yield t, np.asarray(vol[t])

print(f'Anisotropy: {ANISOTROPY:.1f}x')
print('✅ Core structures defined')

In [ ]:
# ── Cell 3: 3D Segmentation Engine ───────────────────────────────────────────────
from skimage.feature import blob_log
from skimage.filters import gaussian
from skimage.measure import regionprops
from skimage.segmentation import relabel_sequential, watershed


def postprocess_labels(labels, min_vol=50, max_vol=50_000):
    """Remove too-small and too-large objects, relabel sequentially."""
    out = labels.copy()
    for r in regionprops(labels):
        if r.area < min_vol or r.area > max_vol:
            out[labels == r.label] = 0
    out, _, _ = relabel_sequential(out)
    return out


class CellSegmenter:
    """3D instance segmenter: Cellpose cyto3 with blob fallback."""
    
    def __init__(self, diameter=12.0, anisotropy=4.0, flow_threshold=0.4,
                 cellprob_threshold=0.0, min_size=50, max_volume=50_000,
                 model_type='cyto3'):
        self.diameter = diameter
        self.anisotropy = anisotropy
        self.flow_threshold = flow_threshold
        self.cellprob_threshold = cellprob_threshold
        self.min_size = min_size
        self.max_volume = max_volume
        self.model_type = model_type
        self._model = None
    
    def _get_model(self):
        if self._model is None:
            from cellpose import models
            self._model = models.CellposeModel(gpu=True, model_type=self.model_type)
            logger.info(f'Loaded Cellpose model: {self.model_type}')
        return self._model
    
    def segment_frame(self, img, frame_idx=0):
        try:
            model = self._get_model()
            masks, _, _ = model.eval(
                img, diameter=self.diameter, channels=[0, 0],
                do_3D=True, anisotropy=self.anisotropy,
                flow_threshold=self.flow_threshold,
                cellprob_threshold=self.cellprob_threshold,
                min_size=self.min_size,
            )
            raw = masks.astype(np.int32)
        except Exception as e:
            logger.warning(f'Cellpose failed (frame {frame_idx}): {e} — using blob fallback')
            raw = self._blob_fallback(img)
        
        labels = postprocess_labels(raw, self.min_size, self.max_volume)
        cells = []
        for r in regionprops(labels):
            c_vox = np.array(r.centroid, dtype=float)
            c_um = c_vox * np.array(VOXEL_SIZE_UM)
            cells.append(Cell(id=int(r.label), frame=frame_idx,
                             centroid=c_vox, centroid_um=c_um, volume=float(r.area)))
        return labels, cells
    
    def _blob_fallback(self, img):
        img_n = img.astype(float)
        img_n -= img_n.min()
        if img_n.max() > 0:
            img_n /= img_n.max()
        sigma_xy = self.diameter / 4.0
        sigma_z = sigma_xy / self.anisotropy
        smoothed = gaussian(img_n, sigma=(sigma_z, sigma_xy, sigma_xy))
        blobs = blob_log(smoothed, min_sigma=max(1.0, self.diameter/8),
                        max_sigma=self.diameter/2, num_sigma=5, threshold=0.05)
        if len(blobs) == 0:
            return np.zeros(img.shape, dtype=np.int32)
        seeds = np.zeros(img.shape, dtype=np.int32)
        for i, (z, y, x, _) in enumerate(blobs):
            zz, yy, xx = int(round(z)), int(round(y)), int(round(x))
            if 0 <= zz < img.shape[0] and 0 <= yy < img.shape[1] and 0 <= xx < img.shape[2]:
                seeds[zz, yy, xx] = i + 1
        return watershed(-smoothed, seeds, mask=smoothed > 0.05).astype(np.int32)

print('✅ CellSegmenter defined (Cellpose cyto3 + blob fallback)')

In [ ]:
# ── Cell 4: Hungarian Linker + Division Detector ─────────────────────────────────
class HungarianLinker:
    def __init__(self, max_distance=10.0, use_volume_cost=True, volume_weight=0.3):
        self.max_distance = max_distance
        self.use_volume_cost = use_volume_cost
        self.volume_weight = volume_weight

    def link(self, cells_t, cells_t1):
        if not cells_t or not cells_t1:
            return []
        c0 = np.array([c.centroid_um if c.centroid_um is not None else c.centroid for c in cells_t])
        c1 = np.array([c.centroid_um if c.centroid_um is not None else c.centroid for c in cells_t1])
        cost = cdist(c0, c1, metric='euclidean')
        if self.use_volume_cost:
            vol_t = np.array([c.volume for c in cells_t])[:, None]
            vol_t1 = np.array([c.volume for c in cells_t1])[None, :]
            vol_ratio = np.minimum(vol_t, vol_t1) / (np.maximum(vol_t, vol_t1) + 1e-8)
            vol_cost = (1.0 - vol_ratio) * self.max_distance
            cost = (1 - self.volume_weight) * cost + self.volume_weight * vol_cost
        cost_masked = cost.copy()
        cost_masked[cost >= self.max_distance] = self.max_distance * 10
        row_ind, col_ind = linear_sum_assignment(cost_masked)
        links = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < self.max_distance:
                conf = 1.0 - cost[r, c] / self.max_distance
                links.append((cells_t[r].id, cells_t1[c].id, float(conf)))
        return links


class DivisionDetector:
    def __init__(self, min_size_ratio=0.3, max_size_ratio=0.8, max_distance_um=10.0):
        self.min_size_ratio = min_size_ratio
        self.max_size_ratio = max_size_ratio
        self.max_distance_um = max_distance_um

    def detect(self, all_cells, links):
        divisions = []
        frames = sorted(all_cells.keys())
        for i in range(len(frames) - 1):
            t, t1 = frames[i], frames[i+1]
            linked_t = {l[1] for l in links if l[0] == t}
            linked_t1 = {l[2] for l in links if l[0] == t}
            orphans_t = [c for c in all_cells[t] if c.id not in linked_t]
            orphans_t1 = [c for c in all_cells[t1] if c.id not in linked_t1]
            if not orphans_t or len(orphans_t1) < 2:
                continue
            for mother in orphans_t:
                cands = [d for d in orphans_t1 if mother.distance_to(d) <= self.max_distance_um]
                if len(cands) >= 2:
                    for i1 in range(len(cands)):
                        for i2 in range(i1+1, len(cands)):
                            d1, d2 = cands[i1], cands[i2]
                            conf = self._score(mother, d1, d2)
                            if conf > 0.5:
                                divisions.append(DivisionEvent(
                                    mother.id, t, d1.id, d2.id, t1, conf))
                                break
                        else:
                            continue
                        break
        return divisions

    def _score(self, mother, d1, d2):
        vol_err = abs(mother.volume - (d1.volume + d2.volume)) / (mother.volume + 1e-8)
        vol_score = 1.0 - min(1.0, vol_err)
        r1 = d1.volume / (mother.volume + 1e-8)
        r2 = d2.volume / (mother.volume + 1e-8)
        sym = 1.0 if (self.min_size_ratio <= r1 <= self.max_size_ratio and
                      self.min_size_ratio <= r2 <= self.max_size_ratio) else 0.0
        avg_dist = (mother.distance_to(d1) + mother.distance_to(d2)) / 2.0
        dist_score = 1.0 - avg_dist / self.max_distance_um
        return float(0.5 * vol_score * sym + 0.5 * dist_score)

print('✅ HungarianLinker + DivisionDetector defined')

In [ ]:
# ── Cell 5: Submission CSV Builder ────────────────────────────────────────────────
HEADER = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']

class SubmissionBuilder:
    def build_rows(self, dataset, all_cells, links, divisions=()):
        node_ids = {}
        rows = []
        nid = 1
        for frame in sorted(all_cells):
            for cell in sorted(all_cells[frame], key=lambda c: c.id):
                node_ids[(frame, cell.id)] = nid
                z, y, x = (int(round(v)) for v in cell.centroid)
                rows.append({'dataset': dataset, 'row_type': 'node', 'node_id': nid,
                            't': frame, 'z': z, 'y': y, 'x': x, 'source_id': -1, 'target_id': -1})
                nid += 1
        
        edge_keys = set()
        edges = []
        for frame, src, tgt, _ in links:
            s_key, t_key = (frame, src), (frame + 1, tgt)
            k = (s_key, t_key)
            if s_key in node_ids and t_key in node_ids and k not in edge_keys:
                edge_keys.add(k)
                edges.append(k)
        
        for div in divisions:
            p_key = (div.parent_frame, div.parent_id)
            for ch in (div.child1_id, div.child2_id):
                c_key = (div.division_frame, ch)
                k = (p_key, c_key)
                if p_key in node_ids and c_key in node_ids and k not in edge_keys:
                    edge_keys.add(k)
                    edges.append(k)
        
        for (sf, sc), (tf, tc) in edges:
            rows.append({'dataset': dataset, 'row_type': 'edge', 'node_id': -1,
                        't': -1, 'z': -1, 'y': -1, 'x': -1,
                        'source_id': node_ids[(sf, sc)], 'target_id': node_ids[(tf, tc)]})
        return rows

    def write(self, rows, output_path):
        output_path = pathlib.Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open('w', newline='', encoding='utf-8') as h:
            writer = csv.DictWriter(h, fieldnames=HEADER)
            writer.writeheader()
            for r_id, row in enumerate(rows):
                writer.writerow({'id': r_id, **row})

print('✅ SubmissionBuilder defined')

In [ ]:
# ── Cell 6: Full Pipeline Execution ──────────────────────────────────────────────
ON_KAGGLE = pathlib.Path('/kaggle').exists()

if ON_KAGGLE:
    TEST_DIR = pathlib.Path('/kaggle/input/biohub-cell-tracking-during-development/test')
    OUTPUT = pathlib.Path('/kaggle/working/submission.csv')
else:
    TEST_DIR = pathlib.Path('../data/test')
    OUTPUT = pathlib.Path('submission.csv')


def find_test_samples(test_dir):
    if not test_dir.exists() and ON_KAGGLE:
        for p in pathlib.Path('/kaggle/input').rglob('test'):
            if p.is_dir():
                test_dir = p
                break
    if not test_dir.exists():
        return []
    return sorted([p for p in test_dir.iterdir() 
                   if p.name.endswith('.zarr') or (p.is_dir() and not p.name.startswith('.'))])


def process_sample(sample_path, segmenter):
    """Run segmentation + tracking on a single sample."""
    all_cells = {}
    for frame_idx, image in iter_frames(sample_path):
        _, cells = segmenter.segment_frame(image, frame_idx)
        all_cells[frame_idx] = cells
        if frame_idx % 5 == 0:
            logger.info(f'  Frame {frame_idx}: {len(cells)} cells detected')
    
    # Primary linking
    linker = HungarianLinker(max_distance=10.0, use_volume_cost=True, volume_weight=0.3)
    links = []
    linked_src = {f: set() for f in all_cells}
    linked_tgt = {f: set() for f in all_cells}
    frames = sorted(all_cells)
    
    for f in frames[:-1]:
        for src, tgt, conf in linker.link(all_cells[f], all_cells[f + 1]):
            links.append((f, src, tgt, conf))
            linked_src[f].add(src)
            linked_tgt[f + 1].add(tgt)
    
    # Gap-2 bridging
    for f in frames[:-2]:
        t2 = f + 2
        if t2 not in all_cells:
            continue
        orphan_s = [c for c in all_cells[f] if c.id not in linked_src[f]]
        orphan_t = [c for c in all_cells[t2] if c.id not in linked_tgt[t2]]
        if orphan_s and orphan_t:
            for src, tgt, conf in linker.link(orphan_s, orphan_t):
                links.append((f, src, tgt, conf * 0.9))
                linked_src[f].add(src)
                linked_tgt[t2].add(tgt)
    
    # Division detection
    divisions = DivisionDetector(max_distance_um=10.0).detect(all_cells, links)
    
    n_cells = sum(len(c) for c in all_cells.values())
    logger.info(f'  Result: {n_cells} cells, {len(links)} links, {len(divisions)} divisions')
    return all_cells, links, divisions


# ── RUN ───────────────────────────────────────────────────────────────────────
segmenter = CellSegmenter(diameter=12.0, anisotropy=ANISOTROPY, model_type='cyto3')
builder = SubmissionBuilder()
all_rows = []

samples = find_test_samples(TEST_DIR)
print(f'Environment: {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'Test directory: {TEST_DIR}')
print(f'Found {len(samples)} test samples')

for sp in samples:
    t0 = time.time()
    logger.info(f'Processing {sp.name}...')
    cells, links, divs = process_sample(sp, segmenter)
    rows = builder.build_rows(sp.stem, cells, links, divs)
    all_rows.extend(rows)
    logger.info(f'  {sp.name}: {len(rows)} rows in {time.time()-t0:.1f}s')

if all_rows:
    builder.write(all_rows, OUTPUT)
    print(f'\n✅ SUCCESS: Wrote {len(all_rows):,} rows to {OUTPUT}')
else:
    print('\n⚠️ No test samples found. Attach the competition dataset.')

In [ ]:
# ── Cell 7: Validation & Preview ─────────────────────────────────────────────────
import pandas as pd

if OUTPUT.exists():
    df = pd.read_csv(OUTPUT)
    
    print('=' * 60)
    print(f'📊 Submission Summary')
    print('=' * 60)
    print(f'Total rows:    {len(df):,}')
    print(f'Node rows:     {(df.row_type == "node").sum():,}')
    print(f'Edge rows:     {(df.row_type == "edge").sum():,}')
    print(f'Datasets:      {df.dataset.nunique()}')
    print(f'Columns:       {list(df.columns)}')
    
    # Schema validation
    expected = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    assert list(df.columns) == expected, f'Schema mismatch!'
    assert df.row_type.isin(['node', 'edge']).all(), 'Invalid row_type!'
    
    nodes = df[df.row_type == 'node']
    edges = df[df.row_type == 'edge']
    
    print(f'\n📈 Per-Dataset Breakdown:')
    for ds in df.dataset.unique():
        ds_df = df[df.dataset == ds]
        n_nodes = (ds_df.row_type == 'node').sum()
        n_edges = (ds_df.row_type == 'edge').sum()
        n_frames = ds_df[ds_df.row_type == 'node'].t.nunique()
        print(f'  {ds}: {n_nodes} nodes, {n_edges} edges, {n_frames} frames')
    
    print(f'\n✅ All schema assertions passed!')
    print(f'\nFirst 10 rows:')
    display(df.head(10))
else:
    print('No submission CSV found.')

---

## 📌 Pipeline Summary

| Component | Configuration |
|-----------|---------------|
| **Segmentation** | Cellpose cyto3, `diameter=12`, `anisotropy=4.0`, `do_3D=True` |
| **Post-processing** | Size filter: 50–50K voxels |
| **Linking** | Hungarian, `max_dist=10µm`, `vol_weight=0.3` |
| **Gap closing** | Gap-2, `penalty=0.9` |
| **Division** | Orphan-pair, volume conservation, `conf>0.5` |

### 🔮 Ideas for Improvement
- **Multi-scale segmentation** — Run Cellpose at multiple diameters, merge results
- **ILP tracking** — Integer Linear Programming for globally optimal tracking
- **Deep feature matching** — Use Cellpose embeddings for linking instead of distance alone
- **Gap-3 bridging** — Close longer gaps with stricter thresholds
- **Adaptive thresholds** — Per-sample parameter tuning based on intensity statistics

---

**If you found this solution helpful, please upvote! 👍**